# 00 · Set up WSL2 for AlgoGauge

Run this once on a fresh WSL2 Ubuntu 24.04. Step 1 needs `sudo`, so it is printed for you to run in a terminal; the rest runs here.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))
import os; os.chdir(ROOT)
print("repo root:", ROOT)

## 1. System packages (run in a terminal, needs your password)

In [ ]:
print("wsl -d Ubuntu -e sudo bash", ROOT / "scripts" / "setup_wsl.sh")

## 2. Verify toolchain

In [ ]:
import shutil, subprocess
for tool in ["g++", "cmake", "ninja", "perf", "uv", "git"]:
    print(f"{tool:<6}", shutil.which(tool) or "MISSING")
paranoid_path = pathlib.Path("/proc/sys/kernel/perf_event_paranoid")
print("perf_event_paranoid =", paranoid_path.read_text().strip() if paranoid_path.exists() else "n/a")

## 3. Submodules + Python deps

In [ ]:
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=True)
subprocess.run(["uv", "sync", "--extra", "dev"], check=True)

## 4. Build benchmarks (Release)

In [ ]:
subprocess.run(["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release"], check=True)
subprocess.run(["cmake", "--build", "build", "--parallel"], check=True)

## 5. Smoke run (no perf)

In [ ]:
from algogauge import manifest, runner
m = manifest.load(ROOT / "algogauge.toml")
res = runner.run_suite(m.suites[0], m.defaults, ROOT, skip_perf=True)
print(res.run_id, len(res.records), "benchmarks recorded")